<a href="https://colab.research.google.com/github/aparna-2001/nifty50-stock-price-forecasting/blob/main/yearly_data_k_fold_cross_validation_bootstrappingipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.utils import resample
import numpy as np

In [6]:
import pandas as pd
from google.colab import files


uploaded = files.upload()

yearly = pd.read_csv('nifty50_yearly_features.csv')
features_m5 = ['HL_Range_%', 'Prev_HL_Range_%', 'Recovery_Rate_%',
                'Prev_Recovery_Rate_%', 'Body_Ratio',
                'Upper_Shadow_%', 'Lower_Shadow_%']
print(yearly.shape)

Saving nifty50_yearly_features.csv to nifty50_yearly_features (3).csv
(25, 16)


In [7]:

X_cv = yearly[features_m5].values
y_cv = yearly['Target'].values

rf_cv = RandomForestClassifier(n_estimators=50, random_state=42)




*k -fold cross validation*

In [8]:
# --- 1. K-Fold (10) ---
kfold = KFold(n_splits=10, shuffle=False)
kfold_scores = cross_val_score(rf_cv, X_cv, y_cv, cv=kfold, scoring='accuracy')
print("10-Fold Cross Validation")
print(f"  Scores : {[round(s*100,1) for s in kfold_scores]}")
print(f"  Mean   : {kfold_scores.mean()*100:.1f}%")
print(f"  Std    : {kfold_scores.std()*100:.1f}%")

10-Fold Cross Validation
  Scores : [np.float64(100.0), np.float64(66.7), np.float64(66.7), np.float64(66.7), np.float64(66.7), np.float64(100.0), np.float64(50.0), np.float64(50.0), np.float64(100.0), np.float64(50.0)]
  Mean   : 71.7%
  Std    : 19.8%


*Leave one out cross validation*

In [9]:

loo = LeaveOneOut()
loo_scores = cross_val_score(rf_cv, X_cv, y_cv, cv=loo, scoring='accuracy')
print("\nLeave One Out Cross Validation")
print(f"  Mean   : {loo_scores.mean()*100:.1f}%")
print(f"  Std    : {loo_scores.std()*100:.1f}%")




Leave One Out Cross Validation
  Mean   : 76.0%
  Std    : 42.7%


*Bootstrapping*

In [11]:
n_iterations = 100
boot_scores = []

for i in range(n_iterations):

    train_idx = np.random.choice(len(X_cv), size=len(X_cv), replace=True)
    oob_idx   = np.array([i for i in range(len(X_cv)) if i not in train_idx])

    if len(oob_idx) == 0:
        continue

    X_boot, y_boot = X_cv[train_idx], y_cv[train_idx]
    X_oob,  y_oob  = X_cv[oob_idx],  y_cv[oob_idx]

    rf_cv.fit(X_boot, y_boot)
    preds = rf_cv.predict(X_oob)
    boot_scores.append(accuracy_score(y_oob, preds))

print("\nBootstrapping (100 iterations)")
print(f"  Mean   : {np.mean(boot_scores)*100:.1f}%")
print(f"  Std    : {np.std(boot_scores)*100:.1f}%")
print(f"  95% CI : {np.percentile(boot_scores,2.5)*100:.1f}% – {np.percentile(boot_scores,97.5)*100:.1f}%")


Bootstrapping (100 iterations)
  Mean   : 73.4%
  Std    : 12.0%
  95% CI : 47.1% – 90.0%


**Summary table**

In [12]:

print("\n| Method         | Mean  | Std   |")
print("|----------------|-------|-------|")
print(f"| 10-Fold CV     | {kfold_scores.mean()*100:.1f}%  | {kfold_scores.std()*100:.1f}%  |")
print(f"| Leave One Out  | {loo_scores.mean()*100:.1f}%  | {loo_scores.std()*100:.1f}%  |")
print(f"| Bootstrapping  | {np.mean(boot_scores)*100:.1f}%  | {np.std(boot_scores)*100:.1f}%  |")
print(f"| Walk Forward   | 75.0%  |  0.0%  |")


| Method         | Mean  | Std   |
|----------------|-------|-------|
| 10-Fold CV     | 71.7%  | 19.8%  |
| Leave One Out  | 76.0%  | 42.7%  |
| Bootstrapping  | 73.4%  | 12.0%  |
| Walk Forward   | 75.0%  |  0.0%  |


## Cross Validation Results - NIFTY 50 Yearly Prediction

| Method         | Mean  | Std   |
|----------------|-------|-------|
| 10-Fold CV     | 71.7% | 19.8% |
| Leave One Out  | 76.0% | 42.7% |
| Bootstrapping  | 73.4% | 12.0% |
| Walk Forward   | 75.0% |  0.0% |

### Interpretation
- All 4 methods agree - true accuracy is around **72-76%**
- Walk Forward is most reliable - respects time order, zero variance
- Leave One Out highest mean (76%) but most unstable (42.7% std)
- Bootstrapping most honest estimate - 100 random trials, low variance
- High std across all methods = small dataset problem (only 25 years)

### Conclusion
> Model consistently achieves ~75% accuracy regardless of validation method.
> This confirms Walk Forward result was not a fluke.
> Accuracy cannot improve further without more data.